# 01: Topic inventory

**Goal:** find out what is actually inside the downloaded bags, before writing any loader code.

Specifically:

1. Which topic carries the Xsens GNSS, and at what rate?
2. Which topic carries the OBD forward speed?
3. Which topic carries the ground-truth reference?
4. Which message types are custom to NavINST, and therefore need registering before they can be decoded?
5. Which topics look useful but must **not** be used, and why?

Nothing here is production code. The loader proper goes in `src/data_loader.py` once these answers are known.

> **Sequence:** `Urban04` Day. The indoor garage sequences were abandoned because they ship no GNSS data at all. See `CLAUDE.md` §4.

## What a ROS bag actually is

A **bag** is a recording file. Inside it is a long, time-ordered sequence of **messages**. Each message carries:

- a **topic**, a named channel such as `/xsens/imu`. Think of it as one sensor's output stream.
- a **message type**, which is a schema, essentially a struct definition, such as `sensor_msgs/msg/Imu`. It tells you how to interpret the packed bytes.
- a **timestamp**. In fact two of them, which matters later (see the notes at the end).

Normally ROS *replays* a bag in real time to feed live software. We do not want that. We want to iterate the file offline and turn it into tables, which is what the `rosbags` library does. It reads ROS1 and ROS2 bags with no ROS installation at all.

### Connections

A **connection** is the bag's record of "a publisher wrote messages of this type on this topic". It holds the topic name, the message type, the number of messages, and, importantly, the full text of the message definition.

Listing connections requires **no decoding**, so we can inventory a bag containing custom message types we cannot yet deserialize. That is why this is the right first step.

One wrinkle: a single topic can have **several connections** (for example if a publisher restarted mid-recording), so the same topic may appear more than once and its message count is split across those rows. We aggregate below.

In [1]:
from pathlib import Path

import pandas as pd
from rosbags.highlevel import AnyReader

# Work out the repository root whether this runs from the repo root or from notebooks/.
REPO = Path.cwd()
if not (REPO / "data").exists():
    REPO = REPO.parent

SEQUENCE = REPO / "data" / "raw" / "Urban04" / "2023-11-01-12-16-16-Day"
BAGS = {name: SEQUENCE / f"{name}.bag" for name in ("gnss", "imus", "reference")}

print(REPO)
for name, bag in BAGS.items():
    size = f"{bag.stat().st_size / 1e6:.1f} MB" if bag.exists() else "MISSING"
    print(f"{name + '.bag':16s} {size}")

/Users/xinyue/repos/imu-gnss-localization
gnss.bag         61.9 MB
imus.bag         351.6 MB
reference.bag    10.4 MB


## The reader

`AnyReader` is the format-agnostic entry point in `rosbags`. It works out whether it is looking at a ROS1 `.bag` or a ROS2 directory and handles either. It takes a **list** of paths, because a recording split across several files can be read as one logical stream. That is useful here, since NavINST splits each run by sensor group.

It is a **context manager**: `with AnyReader([...]) as reader:` opens the file and guarantees it is closed again, whether the block finishes normally or raises. Everything you need must be read inside that block, because afterwards the file is closed and the reader is spent.

The attributes that matter:

| Attribute | Meaning |
|---|---|
| `reader.connections` | the list of connections described above |
| `reader.start_time`, `reader.end_time` | bag extent, in **nanoseconds** since the Unix epoch |
| `reader.duration` | end minus start, also in nanoseconds |
| `reader.messages(connections=...)` | the actual iterator over `(connection, timestamp, rawdata)` |

Each connection has `.topic`, `.msgtype`, `.msgcount` and `.msgdef`. We use the first three now. `.msgdef` becomes important in the next notebook.

In [2]:
def inventory(bag_path: Path, label: str) -> pd.DataFrame:
    """Return one row per (topic, message type) with counts and rates."""
    with AnyReader([bag_path]) as reader:
        duration_s = reader.duration / 1e9
        rows = [
            {"topic": c.topic, "msgtype": c.msgtype, "count": c.msgcount}
            for c in reader.connections
        ]

    df = (
        pd.DataFrame(rows)
        .groupby(["topic", "msgtype"], as_index=False)["count"]
        .sum()  # collapse duplicate connections for the same topic
    )
    df.insert(0, "bag", label)
    df["rate_hz"] = (df["count"] / duration_s).round(1)
    df["custom"] = ~df["msgtype"].str.startswith(
        ("sensor_msgs/", "geometry_msgs/", "std_msgs/", "nav_msgs/")
    )
    df.attrs["duration_s"] = duration_s
    return df.sort_values("topic").reset_index(drop=True)


tables = {name: inventory(path, f"{name}.bag") for name, path in BAGS.items()}
both = pd.concat(tables.values(), ignore_index=True)

duration_s = tables["gnss"].attrs["duration_s"]
print(f"duration: {duration_s:.1f} s  ({duration_s / 60:.2f} min)")

duration: 1065.0 s  (17.75 min)


### `gnss.bag`

The bag that decides whether this project is viable. Look for a GNSS fix topic and check its rate against the 1 to 10 Hz the receiver spec allows.

Expect more than one plausible candidate here, and read the cell after the table before choosing between them.

In [3]:
tables["gnss"]

,bag,topic,msgtype,count,rate_hz,custom
0,gnss.bag,/novatel/gps,gps_common/msg/GPSFix,53250,50.0,True
1,gnss.bag,/xsens/filter/positionlla,geometry_msgs/msg/Vector3Stamped,106500,100.0,False
2,gnss.bag,/xsens/filter/quaternion,geometry_msgs/msg/QuaternionStamped,106500,100.0,False
3,gnss.bag,/xsens/filter/twist,geometry_msgs/msg/TwistStamped,106500,100.0,False
4,gnss.bag,/xsens/gnss,sensor_msgs/msg/NavSatFix,4260,4.0,False


### Two traps in `gnss.bag`

**`/novatel/gps` is ground truth hardware.** It is the NovAtel receiver, the other half of the reference system. It sits in the same bag as the input, at an attractive 50 Hz, and `gps_common/msg/GPSFix` even carries DOP fields. Using it as a filter input would be feeding the answer key into the filter.

**`/xsens/filter/*` is the Xsens's own fused solution.** The MTi-670G is not a bare IMU, it is a GNSS/INS device running its own internal estimator. The `positionlla`, `quaternion` and `twist` topics are the output of *that* filter: a position already fused from inertial and satellite data, at a tempting 100 Hz rather than the raw fix rate.

Using those would not be cheating against ground truth, but it would defeat the point of the project. The deliverable is a sensor fusion filter written from scratch. Taking a vendor's already-fused position as a measurement means the fusion has been done for you, by someone else, with an undisclosed algorithm, and no comparison between an EKF and a UKF downstream of it would mean much.

The honest inputs are the **raw GNSS fix** here and the **raw** `/xsens/imu` from `imus.bag`. This belongs in the README as a stated decision alongside the NovAtel split, because it is exactly the kind of choice a reviewer looks for.

### `imus.bag`

Expect the Xsens MTi-670G at 100 Hz and the OBD2 speed at roughly 16 Hz, plus several IMUs belonging to other sensors, since the Livox LiDAR and both ZED cameras carry their own. `/novatel/imu` is the tactical-grade reference IMU and is also off limits.

In [4]:
tables["imus"]

,bag,topic,msgtype,count,rate_hz,custom
0,imus.bag,/livox/imu,sensor_msgs/msg/Imu,215449,202.3,False
1,imus.bag,/novatel/imu,sensor_msgs/msg/Imu,213006,200.0,False
2,imus.bag,/obd2/maf,navinst_obd_driver/msg/ObdMeasurmentStamped,17751,16.7,True
3,imus.bag,/obd2/rpm,navinst_obd_driver/msg/ObdMeasurmentStamped,17751,16.7,True
4,imus.bag,/obd2/speed,navinst_obd_driver/msg/ObdMeasurmentStamped,17751,16.7,True
5,imus.bag,/xsens/imu,sensor_msgs/msg/Imu,106500,100.0,False
6,imus.bag,/xsens/mag,geometry_msgs/msg/Vector3Stamped,106500,100.0,False
7,imus.bag,/xsens/pressure,sensor_msgs/msg/FluidPressure,106500,100.0,False
8,imus.bag,/xsens/temperature,sensor_msgs/msg/Temperature,106500,100.0,False
9,imus.bag,/zedx_left/imu,sensor_msgs/msg/Imu,106432,99.9,False


### `reference.bag`

Ground truth only. For urban sequences this is the post-processed NovAtel GNSS/INS solution, PPK corrected to roughly 1 cm for Urban01 to Urban04. Unlike the indoor LiDAR map reference, it does **not** fuse the OBD odometer, so it shares nothing with the filter inputs.

In [5]:
tables["reference"]

,bag,topic,msgtype,count,rate_hz,custom
0,reference.bag,/novatel/reference/postprocessed/inspva,navinst_ros/msg/INSPVA,53251,50.0,True
1,reference.bag,/novatel/reference/postprocessed/inspvax,navinst_ros/msg/INSPVAX,1066,1.0,True


## Which topics does the project actually need?

From `CLAUDE.md` §4, and the hard rule that the NovAtel and KVH hardware is **ground truth only and never a filter input**:

| Role | Expected source | Expected rate |
|---|---|---|
| Input, inertial | Xsens MTi-670G, raw | 100 Hz |
| Input, absolute position | Xsens GNSS, raw fix | 1–10 Hz |
| Input, wheel speed | OBD-II vehicle speed | 16 Hz |
| Ground truth only | NovAtel post-processed | 50 Hz |

Everything else is out of scope: the Livox and ZED IMUs, pressure, temperature, magnetometer, RPM, MAF, the Xsens fused solution, and anything NovAtel. The magnetometer is excluded deliberately, per §3. An urban canyon is full of steel and reinforced concrete, which distorts the local field and invalidates hard-iron calibration, so it would inject a confident wrong heading in exactly the places where GNSS is already failing.

In [6]:
# The streams that matter, and nothing else.
keep = ["/xsens/gnss", "/xsens/imu", "/obd2/speed"]

selected = both[
    both["topic"].isin(keep) | both["topic"].str.contains("reference", case=False)
]
selected[["bag", "topic", "msgtype", "count", "rate_hz", "custom"]]

,bag,topic,msgtype,count,rate_hz,custom
4,gnss.bag,/xsens/gnss,sensor_msgs/msg/NavSatFix,4260,4.0,False
9,imus.bag,/obd2/speed,navinst_obd_driver/msg/ObdMeasurmentStamped,17751,16.7,True
10,imus.bag,/xsens/imu,sensor_msgs/msg/Imu,106500,100.0,False
18,reference.bag,/novatel/reference/postprocessed/inspva,navinst_ros/msg/INSPVA,53251,50.0,True
19,reference.bag,/novatel/reference/postprocessed/inspvax,navinst_ros/msg/INSPVAX,1066,1.0,True


In [7]:
# Every custom type present across the three bags. These need registering with
# rosbags before any of their messages can be decoded.
sorted(both.loc[both["custom"], "msgtype"].unique())

['gps_common/msg/GPSFix',
 'navinst_obd_driver/msg/ObdMeasurmentStamped',
 'navinst_ros/msg/INSPVA',
 'navinst_ros/msg/INSPVAX']

## Notes and gotchas

**The package name in the bag may not match the cloned repository.** The OBD type is declared as `navinst_obd_driver/msg/ObdMeasurmentStamped`, while the definition in `~/repos/dataset` lives under `navinst_ros/msg/`. When registering types manually, the name must match what the *bag* says, or lookup fails. The cleanest way to sidestep this entirely is to register from each connection's own embedded `.msgdef`, since ROS1 bags carry the full definition text internally and so can describe themselves.

**Two timestamps exist per message:** the bag's record time, which is what `reader.messages()` yields, and the sensor's own `header.stamp` inside the message. They differ by driver and buffering latency, which is not constant. **Always use `header.stamp`** for time alignment. Fusing a 100 Hz IMU with a few-hertz GNSS at urban driving speeds means tens of milliseconds of timing error turn straight into position error.

**Message count divided by duration is an average**, not proof of a steady rate. A sensor that dropped out for thirty seconds and ran fast elsewhere averages out to something plausible. For GNSS that distinction is the whole project: the gaps between consecutive fixes, and the stretches where quality collapses, are what E3 measures.

---

### Next

1. Register the custom types and decode one message of each topic of interest. In particular, inspect the GNSS message's quality fields (`status`, `position_covariance`, `position_covariance_type`). That decides which branch of §14's adaptive-$R$ table applies.
2. Plot the GNSS track against the reference and find where the fix degrades. How many complete dropouts are there, and how long are they?
3. Then write `src/data_loader.py` properly.